In [1]:
import torch
import pandas as pd
from torch import nn
from torch import optim
from torch.utils.data import Dataset, DataLoader, random_split

In [2]:
class CustomDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_csv(file_path)
        self.x = df.iloc[:, 0].values
        self.y = df.iloc[:, 1].values
        self.length = len(df)

    def __getitem__(self, index):
        x = torch.FloatTensor([self.x[index] ** 2, self.x[index]])
        y = torch.FloatTensor([self.y[index]])
        return x, y

    def __len__(self):
        return self.length

In [3]:
class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(2, 1)

    def forward(self, x):
        x = self.layer(x)
        return x

In [4]:
dataset = CustomDataset("../datasets/non_linear.csv")
dataset_size = len(dataset)
train_size = int(dataset_size * 0.8)
validation_size = int(dataset_size * 0.1)
test_size = dataset_size - train_size - validation_size

train_dataset, validation_dataset, test_dataset = random_split(dataset, [train_size, validation_size, test_size])
print(f"Training Data Size : {len(train_dataset)}")
print(f"Validation Data Size : {len(validation_dataset)}")
print(f"Testing Data Size : {len(test_dataset)}")

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=True)
validation_dataloader = DataLoader(validation_dataset, batch_size=4, shuffle=True, drop_last=True)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=True, drop_last=True)

Training Data Size : 160
Validation Data Size : 20
Testing Data Size : 20


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CustomModel().to(device)
criterion = nn.MSELoss().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.0001)

In [6]:
for epoch in range(10000):
    cost = 0.0

    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)

        output = model(x)
        loss = criterion(output, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss

    cost = cost / len(train_dataloader)

    if (epoch + 1) % 1000 == 0:
        print(f"Epoch : {epoch+1:4d}, Model : {list(model.parameters())}, Cost : {cost:.3f}")

Epoch : 1000, Model : [Parameter containing:
tensor([[ 3.1078, -1.7056]], requires_grad=True), Parameter containing:
tensor([0.0501], requires_grad=True)], Cost : 0.170
Epoch : 2000, Model : [Parameter containing:
tensor([[ 3.1021, -1.7049]], requires_grad=True), Parameter containing:
tensor([0.3265], requires_grad=True)], Cost : 0.088
Epoch : 3000, Model : [Parameter containing:
tensor([[ 3.1004, -1.7046]], requires_grad=True), Parameter containing:
tensor([0.4372], requires_grad=True)], Cost : 0.075
Epoch : 4000, Model : [Parameter containing:
tensor([[ 3.1010, -1.7046]], requires_grad=True), Parameter containing:
tensor([0.4814], requires_grad=True)], Cost : 0.072
Epoch : 5000, Model : [Parameter containing:
tensor([[ 3.1000, -1.7045]], requires_grad=True), Parameter containing:
tensor([0.4991], requires_grad=True)], Cost : 0.073
Epoch : 6000, Model : [Parameter containing:
tensor([[ 3.0999, -1.7043]], requires_grad=True), Parameter containing:
tensor([0.5062], requires_grad=True)],

In [ ]:
with torch.no_grad():
    model.eval()
    for x, y in validation_dataloader:
        x = x.to(device)
        y = y.to(device)
        
        outputs = model(x)
        print(f"X : {x}")
        print(f"Y : {y}")
        print(f"Outputs : {outputs}")
        print("--------------------")

X : tensor([[16.8100,  4.1000],
        [18.4900,  4.3000],
        [ 6.2500, -2.5000],
        [53.2900,  7.3000]])
Y : tensor([[ 45.5200],
        [ 50.0800],
        [ 24.4800],
        [152.8700]])
Outputs : tensor([[ 45.6457],
        [ 50.5140],
        [ 24.1514],
        [153.3056]])
--------------------
X : tensor([[82.8100,  9.1000],
        [ 7.8400,  2.8000],
        [ 8.4100,  2.9000],
        [ 1.4400,  1.2000]])
Y : tensor([[242.1900],
        [ 20.3700],
        [ 21.1800],
        [  2.8300]])
Outputs : tensor([[241.7707],
        [ 20.0481],
        [ 21.6450],
        [  2.9306]])
--------------------
X : tensor([[68.8900,  8.3000],
        [ 5.7600,  2.4000],
        [84.6400, -9.2000],
        [94.0900, -9.7000]])
Y : tensor([[199.9000],
        [ 13.8100],
        [278.3300],
        [308.5100]])
Outputs : tensor([[199.9723],
        [ 14.2803],
        [278.6360],
        [308.7899]])
--------------------
X : tensor([[ 2.5600, -1.6000],
        [27.0400, -5.2000]